# SCS Indicator Manual Labeling Tool -- Multi-Chicken Edition (Kaggle)

Versi ini untuk kasus **satu foto berisi lebih dari satu ekor ayam**. Bedanya dari versi sebelumnya:

- Sumber unit yang dilabeli sekarang **per-ekor-ayam** (instance), bukan per-file gambar.
- Instance per ekor otomatis dideteksi dari `_annotations.coco.json` bawaan Roboflow: box-box
  yang saling tumpang tindih tinggi (>50%) dan berkategori sama digabung jadi satu ekor
  (mengikuti pola dataset ini: 1 box badan + 1 box kepala per ekor).
- Tiap instance di-crop otomatis dari foto aslinya (pakai box terbesar/badan + sedikit padding),
  jadi kamu menilai 7 indikator SCS untuk satu ekor spesifik, bukan tercampur ayam lain di foto
  yang sama.
- Kategori asli dari Roboflow (`normal` / `low_stress` / `medium_stress` / `high_stress`)
  ditampilkan sebagai **referensi saja** -- keputusan label akhir & 7 indikator tetap manual olehmu.

**Untuk foto BARU yang belum ada bounding box-nya sama sekali** (belum dianotasi di Roboflow):
notebook ini tidak bisa membantu -- box per ekor ayam harus digambar dulu di Roboflow/CVAT
(1 box per ekor, kategori = kondisi ekor itu sendiri), baru foto itu attach ulang ke sini setelah
`_annotations.coco.json` ter-update.

**Cara pakai:** jalankan semua cell (Run All) dalam mode Edit/Interactive, isi `COCO_ROOT` di
config, lalu labeli lewat widget di cell terakhir.


In [ ]:
import os, json, csv, glob
import ipywidgets as widgets
from IPython.display import display, clear_output
from PIL import Image


## 1. Konfigurasi

In [ ]:
# --- GANTI sesuai lokasi dataset COCO kamu di Kaggle ---
# Cek dulu kalau belum yakin: !find /kaggle/input -name "_annotations.coco.json"
COCO_ROOT = "/kaggle/input/stressed-chicken-v19i-coco"   # folder yang berisi train/valid/test
SPLITS = ["train", "valid", "test"]

PROGRESS_PATH = "/kaggle/working/indicators_progress_multi.json"
EXPORT_CSV_PATH = "/kaggle/working/indicators_ground_truth_multi.csv"

INDICATORS = [
    ("posisi_ekor", "Posisi ekor"),
    ("posisi_kepala", "Posisi kepala"),
    ("penutupan_mata", "Penutupan mata"),
    ("pembukaan_paruh", "Pembukaan paruh"),
    ("posisi_sayap", "Posisi sayap"),
    ("postur_kaki", "Postur kaki"),
    ("kondisi_bulu", "Kondisi bulu"),
]
LABEL_OPTIONS = ["not_stress", "stress", "undefined"]
STATE_OPTIONS = ["YA", "TIDAK", "TIDAK_TERLIHAT"]
CROP_PADDING_RATIO = 0.08   # tambahan ruang di tepi box badan, supaya ekor/kaki tidak terpotong
OVERLAP_THRESHOLD = 0.5     # ambang gabung box badan+kepala jadi 1 instance ekor ayam


## 2. Deteksi instance per-ekor-ayam dari bounding box

Untuk tiap gambar: box-box dengan kategori sama & saling tumpang tindih tinggi digabung jadi
satu instance (mewakili satu ekor), representasinya pakai box terbesar (badan). Gambar dengan
cuma 1 box juga otomatis jadi 1 instance.

In [ ]:
def box_xyxy(b):
    x, y, w, h = b
    return x, y, x + w, y + h

def overlap_ratio(small_box, big_box):
    sx1, sy1, sx2, sy2 = box_xyxy(small_box)
    bx1, by1, bx2, by2 = box_xyxy(big_box)
    ix1, iy1 = max(sx1, bx1), max(sy1, by1)
    ix2, iy2 = min(sx2, bx2), min(sy2, by2)
    iw, ih = max(0, ix2 - ix1), max(0, iy2 - iy1)
    inter = iw * ih
    small_area = max(1e-6, (sx2 - sx1) * (sy2 - sy1))
    return inter / small_area

def group_instances(anns):
    anns_sorted = sorted(anns, key=lambda a: a["bbox"][2] * a["bbox"][3], reverse=True)
    used = [False] * len(anns_sorted)
    groups = []
    for i, a in enumerate(anns_sorted):
        if used[i]:
            continue
        used[i] = True
        group = [a]
        for j in range(i + 1, len(anns_sorted)):
            if used[j]:
                continue
            b = anns_sorted[j]
            if b["category_id"] == a["category_id"] and overlap_ratio(b["bbox"], a["bbox"]) > OVERLAP_THRESHOLD:
                used[j] = True
                group.append(b)
        groups.append(group)
    return groups

instances = []  # each: dict(split, file_name, image_path, crop_box, ref_category, n_boxes_in_group)

for split in SPLITS:
    ann_path = os.path.join(COCO_ROOT, split, "_annotations.coco.json")
    if not os.path.exists(ann_path):
        print(f"[!] Tidak ditemukan: {ann_path} -- dilewati")
        continue
    d = json.load(open(ann_path))
    cats = {c["id"]: c["name"] for c in d["categories"]}
    imgs = {im["id"]: im for im in d["images"]}
    from collections import defaultdict
    by_img = defaultdict(list)
    for a in d["annotations"]:
        by_img[a["image_id"]].append(a)

    for img_id, anns in by_img.items():
        img_info = imgs[img_id]
        groups = group_instances(anns)
        for gi, group in enumerate(groups):
            rep = max(group, key=lambda a: a["bbox"][2] * a["bbox"][3])
            instances.append({
                "split": split,
                "file_name": img_info["file_name"],
                "image_path": os.path.join(COCO_ROOT, split, img_info["file_name"]),
                "img_w": img_info["width"],
                "img_h": img_info["height"],
                "crop_box": rep["bbox"],
                "ref_category": cats[rep["category_id"]],
                "n_boxes_in_group": len(group),
                "instance_key": f"{split}/{img_info['file_name']}#inst{gi}",
            })

instances.sort(key=lambda r: r["instance_key"])
print(f"Total instance ekor ayam terdeteksi: {len(instances)}  (dari total gambar dengan anotasi)")
multi_chicken_images = sum(1 for r in instances if r["n_boxes_in_group"] >= 1) 
from collections import Counter
per_file = Counter(r["file_name"] for r in instances)
n_multi_files = sum(1 for v in per_file.values() if v > 1)
print(f"Gambar dengan >1 ekor ayam terdeteksi: {n_multi_files} file")


## 3. Muat progres sebelumnya (kalau ada)

In [ ]:
def blank_entry():
    return {"indicators": {k: None for k, _ in INDICATORS}, "label": "", "notes": "", "excluded": False}

if os.path.exists(PROGRESS_PATH):
    with open(PROGRESS_PATH, "r") as f:
        progress = json.load(f)
    print(f"Progres sebelumnya dimuat: {len(progress)} entri.")
else:
    progress = {}
    print("Belum ada progres -- mulai dari nol.")

def get_entry(key):
    if key not in progress:
        progress[key] = blank_entry()
    return progress[key]

def save_progress():
    with open(PROGRESS_PATH, "w") as f:
        json.dump(progress, f, indent=2)


## 4. Widget labeling per-ekor-ayam

In [ ]:
state = {"idx": 0}

img_output = widgets.Output()
fname_label = widgets.Label()
ref_label = widgets.Label()
progress_label = widgets.Label()

indicator_widgets = {}
indicator_rows = []
for key, label in INDICATORS:
    tb = widgets.ToggleButtons(options=STATE_OPTIONS, description=label, style={"description_width": "150px"})
    indicator_widgets[key] = tb
    indicator_rows.append(tb)

label_dropdown = widgets.Dropdown(options=LABEL_OPTIONS, description="Label akhir:", style={"description_width": "150px"})
notes_box = widgets.Text(description="Catatan:", placeholder="opsional", style={"description_width": "150px"}, layout=widgets.Layout(width="500px"))
suggest_label = widgets.Label()

prev_btn = widgets.Button(description="<- Sebelumnya")
next_btn = widgets.Button(description="Berikutnya ->")
exclude_btn = widgets.Button(description="Exclude instance ini", button_style="danger")
export_btn = widgets.Button(description="Export CSV", button_style="success")
export_output = widgets.Output()

_suppress = {"on": False}

def get_crop(inst):
    im = Image.open(inst["image_path"]).convert("RGB")
    x, y, w, h = inst["crop_box"]
    pad_x, pad_y = w * CROP_PADDING_RATIO, h * CROP_PADDING_RATIO
    x1 = max(0, x - pad_x)
    y1 = max(0, y - pad_y)
    x2 = min(inst["img_w"], x + w + pad_x)
    y2 = min(inst["img_h"], y + h + pad_y)
    return im.crop((x1, y1, x2, y2))

def render_current():
    if not instances:
        return
    _suppress["on"] = True
    idx = state["idx"]
    inst = instances[idx]
    key = inst["instance_key"]
    entry = get_entry(key)

    with img_output:
        clear_output(wait=True)
        crop = get_crop(inst)
        crop.thumbnail((450, 450))
        display(crop)

    fname_label.value = f"{inst['split']}/{inst['file_name']}"
    ref_label.value = f"Kategori asli Roboflow (referensi saja): {inst['ref_category']} | box digabung dari {inst['n_boxes_in_group']} anotasi"
    progress_label.value = f"Instance {idx+1} / {len(instances)}" + ("  [DIKECUALIKAN]" if entry["excluded"] else "")

    for key_ind, _ in INDICATORS:
        indicator_widgets[key_ind].value = entry["indicators"][key_ind] or STATE_OPTIONS[1]
    label_dropdown.value = entry["label"] or LABEL_OPTIONS[0]
    notes_box.value = entry["notes"] or ""

    ya_count = sum(1 for k, _ in INDICATORS if entry["indicators"][k] == "YA")
    filled = all(entry["indicators"][k] is not None for k, _ in INDICATORS)
    suggest_label.value = f"Total YA saat ini: {ya_count}/7" if filled else "(belum semua indikator disentuh)"

    prev_btn.disabled = idx == 0
    next_btn.disabled = idx == len(instances) - 1
    _suppress["on"] = False

def commit_current(*_):
    if _suppress["on"] or not instances:
        return
    inst = instances[state["idx"]]
    entry = get_entry(inst["instance_key"])
    for k, _lbl in INDICATORS:
        entry["indicators"][k] = indicator_widgets[k].value
    entry["label"] = label_dropdown.value
    entry["notes"] = notes_box.value
    save_progress()
    ya_count = sum(1 for k, _ in INDICATORS if entry["indicators"][k] == "YA")
    suggest_label.value = f"Total YA saat ini: {ya_count}/7 (tersimpan otomatis)"

for w in indicator_rows:
    w.observe(commit_current, names="value")
label_dropdown.observe(commit_current, names="value")
notes_box.observe(commit_current, names="value")

def on_prev(_):
    commit_current()
    if state["idx"] > 0:
        state["idx"] -= 1
        render_current()

def on_next(_):
    commit_current()
    if state["idx"] < len(instances) - 1:
        state["idx"] += 1
        render_current()

def on_exclude(_):
    inst = instances[state["idx"]]
    entry = get_entry(inst["instance_key"])
    entry["excluded"] = not entry["excluded"]
    save_progress()
    render_current()

def on_export(_):
    commit_current()
    with open(EXPORT_CSV_PATH, "w", newline="") as f:
        writer = csv.writer(f)
        header = ["instance_key", "split", "file_name", "ref_category_roboflow"] + [k for k, _ in INDICATORS] + ["label_final", "notes", "excluded"]
        writer.writerow(header)
        for inst in instances:
            entry = get_entry(inst["instance_key"])
            row = [inst["instance_key"], inst["split"], inst["file_name"], inst["ref_category"]] + \
                  [entry["indicators"][k] or "" for k, _ in INDICATORS] + \
                  [entry["label"], entry["notes"], int(entry["excluded"])]
            writer.writerow(row)
    with export_output:
        clear_output(wait=True)
        n_done = sum(1 for e in progress.values() if all(e["indicators"].values()))
        print(f"Tersimpan ke: {EXPORT_CSV_PATH}")
        print(f"{n_done}/{len(instances)} instance ekor ayam sudah lengkap 7 indikatornya.")

prev_btn.on_click(on_prev)
next_btn.on_click(on_next)
exclude_btn.on_click(on_exclude)
export_btn.on_click(on_export)

ui = widgets.VBox([
    progress_label,
    img_output,
    fname_label,
    ref_label,
    widgets.VBox(indicator_rows),
    label_dropdown,
    suggest_label,
    notes_box,
    widgets.HBox([prev_btn, next_btn, exclude_btn]),
    widgets.HBox([export_btn]),
    export_output,
])

render_current()
display(ui)


## 5. Catatan penting

- `instance_key` formatnya `split/file_name#instN` -- satu file gambar bisa muncul beberapa kali
  di CSV kalau ada beberapa ekor ayam di dalamnya. Ini kunci gabung ke pipeline, bukan lagi
  `file_name` saja.
- Kolom `ref_category_roboflow` cuma referensi dari anotasi asli, BUKAN dianggap ground truth --
  keputusan `label_final` & 7 indikator tetap murni dari penilaian manualmu.
- Kalau heuristik grouping salah kelompokkan box (jarang, tapi mungkin di foto yang sangat padat
  ayamnya), kamu bisa lihat berapa box tergabung lewat teks referensi di atas gambar, dan tandai
  `excluded=1` + catat di notes kalau crop-nya tidak mewakili satu ekor dengan bersih -- lebih
  aman dikecualikan daripada dipaksa dilabeli dari crop yang salah.
